In [ ]:
# ===================================================================================
# CELL 1: CONSOLIDATED ENVIRONMENT SETUP
# This single block installs all required libraries with compatible versions.
# ===================================================================================
!pip install -q --upgrade \
    peft \
    bitsandbytes \
    torch \
    transformers \
    accelerate \
    datasets \
    trl \
    timm \
    Pillow

accelerate>=1.4.0

datasets>=3.0.0

transformers>=4.56.1

In [ ]:
import accelerate
import datasets
import torch
import transformers
import bitsandbytes

print(accelerate.__version__)
print(datasets.__version__)
print(transformers.__version__)

import time
import random
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

In [ ]:
from peft import LoraConfig, TaskType

peft_config = LoraConfig(task_type=TaskType.SEQ_2_SEQ_LM, inference_mode=False, r=8, lora_alpha=32, lora_dropout=0.1)

# No error
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # Use "CAUSAL_LM" for GPT-style LLMs!
    inference_mode=False,            # Important! Must be False for training
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
)

In [ ]:
from kaggle_secrets import UserSecretsClient
# Step 2: Authenticate with Hugging Face
# This is required to download gated models like Llama 3
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HUGGING_FACE_TOKEN")
except Exception as e:
    print("Could not retrieve Hugging Face token. Please ensure it is stored as a Kaggle secret named 'HUGGING_FACE_TOKEN'.")
    # You can manually paste your token here for local testing if needed:
    # hf_token = "YOUR_HF_TOKEN"
    hf_token = None

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Your model config:
model_id = "arcee-ai/Arcee-VyLinh"  # substitute another if needed, e.g., "Qwen/Qwen2-7B"

# Test quantization config (as in your script)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # or use torch.bfloat16 after importing torch
    bnb_4bit_use_double_quant=False
)

try:
    print("Loading model and tokenizer…")
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=hf_token,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("🚀 SUCCESS: Model loaded with 4-bit quantization.")
except Exception as e:
    print("\n❌ FAILED:", type(e).__name__, e)

In [ ]:
from datasets import load_dataset

dataset_id = "tmnam20/ViMedAQA"
EVAL_FULL_DATASET = False

# seed_num = 1
# NUM_SAMPLES_INITIAL = 20

seed_num = 6
NUM_SAMPLES_INITIAL = 1

# seed_num = 3
# NUM_SAMPLES_INITIAL = 20

ENABLE_SUBSET_SAMPLING = False
NUM_SAMPLES_FINAL = 50

ENABLE_SINGLE_INDEX_SELECTION = False
TARGET_INDEX = 0 # The index (starting from 0) of the sample to select.

# Step 4: Load and Prepare the Dataset
try:
    dataset = load_dataset(dataset_id, split="test")
    print(f"Dataset loaded successfully! Total samples: {len(dataset)}")

    if EVAL_FULL_DATASET:
        eval_dataset = dataset
    else:
        # --- First Sampling Step: Always get the initial samples ---
        random.seed(seed_num) # for reproducibility
        initial_random_indices = random.sample(range(len(dataset)), NUM_SAMPLES_INITIAL)
        initial_eval_dataset = dataset.select(initial_random_indices)
    
        print(f"Created an initial random evaluation set with {len(initial_eval_dataset)} samples.")
    
        # --- Conditional second sampling/selection ---
        if ENABLE_SUBSET_SAMPLING:
            print("Subset sampling is ENABLED. Performing second randomization...")
            # Re-seed to ensure this step is also reproducible
            random.seed(seed_num)
            final_random_indices = random.sample(range(len(initial_eval_dataset)), NUM_SAMPLES_FINAL)
            # Final dataset is the smaller, 50-sample subset
            eval_dataset = initial_eval_dataset.select(final_random_indices)
            print(f"Further randomized and reduced the set to a final size of {len(eval_dataset)} samples.")
            
        elif ENABLE_SINGLE_INDEX_SELECTION:
            print(f"Single index selection is ENABLED. Subsetting to 1 sample at index: {TARGET_INDEX}...")
            # Ensure the target index is valid
            if 0 <= TARGET_INDEX < len(initial_eval_dataset):
                # Final dataset is the single selected sample
                eval_dataset = initial_eval_dataset.select([TARGET_INDEX])
                print(f"Successfully created a final dataset with {len(eval_dataset)} sample.")
            else:
                raise IndexError(f"TARGET_INDEX {TARGET_INDEX} is out of bounds for the initial sample size of {len(initial_eval_dataset)}.")

        else:
            # Final dataset is the larger, initial sample set
            eval_dataset = initial_eval_dataset

except Exception as e:
    print(f"Failed to load or process the dataset. Error: {e}")
    eval_dataset = None

In [ ]:
def format_instruction(sample):
	return f"""### Instruction
Use the user's question to respectfully answer in Vietnamese.

### Input
{sample['question']}

### Response
{sample['answer']}
"""


# Step 5: Setup training
try:
    # --- Create a single SFTConfig object for all arguments ---
    sft_config = SFTConfig(
        # Arguments that were in TrainingArguments
        output_dir="arcee-vylinh-finetuned-vimedaqa",
        num_train_epochs=1, # 3
        per_device_train_batch_size=4,
        logging_dir='./logs',
        logging_steps=10,
        learning_rate=2e-4,
        max_length=512,
        save_strategy="no",
    )

    # --- Initialize the Trainer ---
    # Pass the single config object to the 'args' parameter
    trainer = SFTTrainer(
        model=model,
        train_dataset=eval_dataset,
        peft_config=peft_config,
        # tokenizer=tokenizer,
        args=sft_config, # <-- Use the SFTConfig object here
        formatting_func=format_instruction,
    )

    # --- Start timing and training ---
    print("\nStarting model training...")
    start_time = time.time()
    trainer.train()
    end_time = time.time()
    print("...Training finished!")

    # --- Calculate and print the duration ---
    training_duration = end_time - start_time
    print(f"\nTotal training time: {training_duration:.2f} seconds")
    print(f"({training_duration / 60:.2f} minutes)")

except Exception as e:
    print(f"An error occurred during training setup or execution. Error: {e}")

In [ ]:
print("--- Merging LoRA adapters and unloading PEFT ---")

# The trainer.model is currently a PeftModel
print(f"Model type before merging: {type(trainer.model)}")

# Use merge_and_unload to get the final model
merged_model = trainer.model.merge_and_unload()

# The returned object is now a standard transformers model
print(f"Model type after merging: {type(merged_model)}")
print("✅ Adapters merged and PEFT model unloaded.")

# SFT Trainer

In [ ]:
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

# 1. Load the full dataset first
full_dataset = load_dataset("trl-lib/Capybara", split="train")

# 2. Select only the first sample (at index 0)
single_sample_dataset = full_dataset.select([0])

trainer = SFTTrainer(
    model="Qwen/Qwen3-0.6B",
    train_dataset=single_sample_dataset,
    args=SFTConfig(
        output_dir="quick_test_output",  # You MUST provide an output directory
        num_train_epochs=1,               # Only run one time over the data
        save_strategy="no",               # CRITICAL: Disable the slow final save
    )
)
trainer.train()